# GSMA RAG — Quickstart Notebook

**Open Telco RAG**

This notebook walks through the Open Telco RAG pipeline for the **GSMA** corpus —
files spanning multiple document series.

By the end of this notebook you will have:
- Ingested GSMA documents into a local ChromaDB
- Discovered cross-standard alignment between GSMA, CAMARA, and TM Forum
- Asked questions about Open Gateway programme policy
- Evaluated retrieval quality with the evaluation metrics

---

### Prerequisites

```bash
git clone https://github.com/Znbne/telcolens
cd open-telco-rag
pip install -e .
```

GSMA documents require GSMA membership access. Place documents in `data/gsma/`.

Set your API keys in a `.env` file:
```
# For answer generation (choose one or both)
ANTHROPIC_API_KEY=sk-ant-...        # Claude models direct
OPENROUTER_API_KEY=sk-or-...        # Any model via OpenRouter
```

## 1. Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from collections import Counter
import statistics

load_dotenv()

DATA_DIR   = Path("data/gsma")
DB_PATH    = "chroma_db"
MODEL_KEY  = "minilm"
COLLECTION = f"gsma_{MODEL_KEY}"

print(f"Data directory : {DATA_DIR}")
print(f"ChromaDB path  : {DB_PATH}")
print(f"Embedding model: {MODEL_KEY}")
print(f"Collection     : {COLLECTION}")

## 2. Check available documents

The GSMA corpus spans 15 series folders. For the Open Telco AI Initiative,
the most relevant series are **OPG** (Open Gateway) and **FASG** (Fraud and Security).

The corpus also includes RCS, eSim, TSG, IG, ISAG, SAS, AID, WAS, NG, PQTN,
SAM, IDS, and Working Groups documents.

In [ ]:
if DATA_DIR.exists():
    series_dirs = [d for d in DATA_DIR.rglob("*") if d.is_dir() and d.parent == DATA_DIR or
                   d.parent.name == "Raw"]
    pdf_files   = list(DATA_DIR.rglob("*.pdf"))
    docx_files  = list(DATA_DIR.rglob("*.docx"))
    print(f"PDF files  : {len(pdf_files)}")
    print(f"DOCX files : {len(docx_files)}")
    print(f"Total      : {len(pdf_files) + len(docx_files)}")
    print("\nKey series:")
    raw = DATA_DIR / "Raw"
    if raw.exists():
        for d in sorted(raw.iterdir()):
            if d.is_dir():
                n = len(list(d.rglob("*.pdf"))) + len(list(d.rglob("*.docx")))
                print(f"  {d.name:<40} {n} files")
else:
    print(f"[INFO] {DATA_DIR} not found.")
    print("GSMA documents require GSMA membership access.")

## 3. Ingest into ChromaDB

The GSMA ingest pipeline:
1. Loads PDF and DOCX files from all series folders
2. Extracts series identifier from the file path (OPG, FASG, RCS, etc.)
3. Applies paragraph-based chunking with overlap
4. Stores with series metadata for filtered retrieval

In [ ]:
from open_telco_rag.sources.gsma import GSMASource
from open_telco_rag.embedders.sentense_transformers import SentenceTransformerEmbedder
from open_telco_rag.stores.chroma import ChromaStore
from open_telco_rag.core.pipeline import ingest

source   = GSMASource()
embedder = SentenceTransformerEmbedder(model_key=MODEL_KEY)
store    = ChromaStore(path=DB_PATH, collection_name=COLLECTION)

print(f"Starting ingest — {store.count()} chunks already in collection")

if DATA_DIR.exists():
    stats = ingest(
        source     = source,
        store      = store,
        embedder   = embedder,
        data_dir   = str(DATA_DIR),
        chunk_size = 1024,
        overlap    = 200,
        resume     = True,
    )
    print(f"\nIngest complete")
    print(f"  Files processed : {stats.files_processed}")
    print(f"  Chunks stored   : {stats.chunks_stored}")
    print(f"  Skipped         : {stats.skipped_files}")
    print(f"  Errors          : {stats.error_files}")
print(f"  Total in DB     : {store.count():,}")

## 4. Explore the Collection

In [ ]:
import chromadb

client     = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection(COLLECTION)
sample     = collection.get(limit=2000, include=["documents", "metadatas"])

series_counts = Counter(m.get("series", "unknown") for m in sample["metadatas"])
lengths       = [len(d) for d in sample["documents"]]

print(f"Collection: {COLLECTION}")
print(f"Total chunks: {collection.count():,}\n")
print(f"Series distribution (sample of 2,000 chunks):")
print(f"{'Series':<30} {'Chunks':>8}")
print("-" * 40)
for series, count in series_counts.most_common(15):
    print(f"{series:<30} {count:>8}")

print(f"\nChunk length statistics (chars):")
print(f"  Mean   : {statistics.mean(lengths):.0f}")
print(f"  Median : {statistics.median(lengths):.0f}")

## 5. Semantic Search

GSMA OPG.10 is the key cross-standard reference document — it maps the
relationships between GSMA Open Gateway, CAMARA, and TM Forum TMF931.

In [ ]:
from open_telco_rag.core.pipeline import search

QUERY = "What is the GSMA Open Gateway and how does it relate to CAMARA?"
TOP_K = 5

results = search(
    QUERY,
    store    = store,
    embedder = embedder,
    top_k    = TOP_K,
)

print(f"Query: '{QUERY}'")
print(f"Top-{TOP_K} results:\n")
print(f"{'Rank':<5} {'Similarity':>10} {'Series':<10} {'Source (truncated)'}")
print("-" * 75)

for i, r in enumerate(results, 1):
    sim    = round(1 - r.distance, 3)
    series = r.metadata.get("series", "")
    src    = r.metadata.get("source", "")[:45]
    print(f"{i:<5} {sim:>10.3f} {series:<10} {src}")

print(f"\nTop result preview:")
print(results[0].text[:400])

### 5.1 Cross-standard alignment discovery

GSMA OPG.10 is unique: it explicitly maps three standards bodies' outputs.
A single retrieved document surfaces CAMARA, TM Forum TMF931, and GSMA
Open Gateway as aligned specifications.

In [ ]:
from open_telco_rag.generators.openrouter import OpenRouterGenerator
generator = OpenRouterGenerator(model="meta-llama/llama-3.3-70b-instruct")

QUERY2 = "How does GSMA OPG.10 describe the alignment between Open Gateway and TM Forum?"

results2  = search(QUERY2, store=store, embedder=embedder, top_k=5)

from open_telco_rag.core.pipeline import generate
response2 = generate(QUERY2, results2, source_type="gsma", generator=generator)

print(f"Query: '{QUERY2}'")
print(f"\nAnswer:\n{response2['answer']}")
print(f"\nSources: {response2['sources']}")

## 6. Question Answering

In [ ]:
from open_telco_rag.core.pipeline import generate

QUESTION = "What is the GSMA Open Gateway and how does it relate to CAMARA?"

retrieved = search(QUESTION, store=store, embedder=embedder, top_k=5)

from open_telco_rag.generators.openrouter import OpenRouterGenerator
generator = OpenRouterGenerator(model="meta-llama/llama-3.3-70b-instruct")

response  = generate(QUESTION, retrieved, source_type="gsma", generator=generator)

print(f"Question: {QUESTION}")
print(f"{'─'*70}")
print(f"\n{response['answer']}")
print(f"\n{'─'*70}")
print(f"Sources used: {response['sources']}")
print(f"Chunks used : {response['chunks_used']}")

### 6.1 Question bank

In [ ]:
QUESTIONS = [
    ("What is the GSMA Open Gateway and how does it relate to CAMARA?",             "OPG"),
    ("How does GSMA OPG.10 describe the alignment between Open Gateway and TM Forum?", "OPG"),
    ("What are the GSMA fraud and security specifications?",                          "FASG"),
    ("What does the GSMA define for network API monetisation?",                       "OPG"),
    ("How does GSMA define roaming and interconnect security requirements?",           "FASG"),
]

print(f"{'#':<3} {'Expected series':<18} {'Top retrieved':<18} {'Mean sim':>9} {'Question'}")
print("-" * 90)

for i, (question, expected_series) in enumerate(QUESTIONS, 1):
    results  = search(question, store=store, embedder=embedder, top_k=5)
    sims     = [1 - r.distance for r in results]
    mean_sim = sum(sims) / len(sims) if sims else 0
    top_s    = results[0].metadata.get('series', '?') if results else '?'
    hit      = "✓" if expected_series.lower() in top_s.lower() else "✗"
    q_short  = question[:45] + "..." if len(question) > 45 else question
    print(f"{i:<3} {expected_series:<18} {top_s + ' ' + hit:<18} {mean_sim:>9.3f} {q_short}")

## 7. Evaluation

In [ ]:
from open_telco_rag.evaluation.evaluate import evaluate_corpus
from open_telco_rag.evaluation.questions import load_questions

questions = load_questions("gsma")
print(f"Loaded {len(questions)} evaluation questions for GSMA")
print(f"Questions with reference answers: {sum(1 for q in questions if q.reference_answer)}")

In [ ]:
from open_telco_rag.evaluation._claude import make_llm_call

# RAG generation model (answers questions)
rag_llm_call = make_llm_call("openrouter", "meta-llama/llama-3.3-70b-instruct")

# Metrics evaluation model (scores faithfulness, relevancy, etc.)
llm_call = make_llm_call("openrouter", "anthropic/claude-sonnet-4.5")

result = evaluate_corpus(
    corpus       = "gsma",
    store        = store,
    embedder     = embedder,
    questions    = questions,
    model        = MODEL_KEY,
    top_k        = 5,
    delay        = 1.0,
    llm_call     = llm_call,
    rag_llm_call = rag_llm_call,
)


In [ ]:
print(f"\nPer-question results — GSMA ({MODEL_KEY}, top-k=5)")
print(f"{'#':<3} {'Faith':>6} {'Relev':>6} {'Sim':>6} {'Recall':>7} {'Correct':>8} {'Question'}")
print("-" * 95)

for i, qr in enumerate(result.question_results, 1):
    q_short = qr.question[:45] + "..." if len(qr.question) > 45 else qr.question
    print(
        f"{i:<3} "
        f"{qr.faithfulness:>6.2f} "
        f"{qr.answer_relevancy:>6.2f} "
        f"{qr.mean_similarity:>6.3f} "
        f"{max(0, qr.context_recall):>7.2f} "
        f"{max(0, qr.answer_correctness):>8.2f} "
        f"{q_short}"
    )

print("-" * 95)
print(
    f"{'AVG':<3} "
    f"{result.mean_faithfulness:>6.2f} "
    f"{result.mean_answer_relevancy:>6.2f} "
    f"{result.mean_similarity:>6.3f} "
    f"{max(0, result.mean_context_recall):>7.2f} "
    f"{max(0, result.mean_answer_correctness):>8.2f}"
)


## 8. What the metrics tell us

| Metric | Type | Interpretation |
|--------|------|----------------|
| **Faithfulness** | Generation | Are the claims in the answer grounded in the retrieved chunks? |
| **Answer relevancy** | Generation | Does the answer address what was asked? |
| **Mean similarity** | Retrieval | How confident is the retrieval? Higher = better chunk matches |
| **Context recall** | Retrieval+GT | Were the chunks needed to answer the question retrieved? |
| **Answer correctness** | Generation+GT | Is the answer factually correct vs the reference answer? |
